### Phase 2. Part1 (dummy classifier + baseline model)

Build a dummy classifier and a first real, explainable baseline for pixel-level anomaly
detection, following Report 1's metric decisions.

### Roadmap

* Load and preprocess images per category (resize, grayscale/RGB handling, normalize to [0, 1]).
* Split normal training images into a fit set and a held-out validation set. Validation is
  only used to pick the threshold, never for training or testing.
* Dummy classifier ("always normal") + evaluation, to show the accuracy paradox under severe
  pixel imbalance.
* First real baseline: per-pixel z-score vs. normal training stats. Simple, fully explainable,
  no deep learning, different from the deep-feature nearest-neighbor models like ResNet-18 + Nearest Neighbors and PatchCore.
* Threshold picked from held-out normal validation data only (99th percentile), never from
  the test set.
* Metric: pixel-level Average Precision (PR-AUC), per Report 1 Appendix D. AUROC is not used,
  since Report 1 found it unreliable under this much pixel imbalance.
* Run on two categories (screw: grayscale, bottle: RGB) to see how the baseline's performance
  changes by category and image type.


#### Imports + Checking All 15 categories loading correctly:

In [281]:
# (imports + categories)

from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image # (Pillow) is a Python library for opening, reading, and manipulating image files
from sklearn.metrics import average_precision_score, roc_auc_score, accuracy_score

DATA_ROOT = Path("../../data/raw/mvtec_ad")
categories = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir() and (d / "test").exists()])
print(categories)

['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']


#### Scope and constants:

In [282]:

# scope decision (which categories we are actually testing) and a couple of constants, SIZE, RANDOM_SEED, etc.

SIZE = 256  # working resolution, square
RANDOM_SEED = 42
VAL_FRACTION = 0.15  # slice of train_good held out ONLY for picking a threshold later

PHASE2_CATEGORIES = ["screw", "metal_nut", "tile", "grid"]
GRAYSCALE_CATEGORIES = {"grid", "screw", "tile", "toothbrush", "zipper"}

Why these values: 
- SIZE = 256: 
it is the pixel resolution that will be used to resize every image to, so they're all directly comparable (raw images vary between 700–1024 pixels, too big and inconsistent to compare pixel-by-pixel as-is). 
- PHASE2_CATEGORIES: 
it is the 4-category scope from EDA, screw and metal_nut are the extremes of pixel-level imbalance (0.34% vs 14.5%, the biggest split was  found) and tile and grid add two texture categories with different defect shapes. 
- GRAYSCALE_CATEGORIES:
it matches what EDA already confirmed, those 5 categories are natively grayscale, not RGB, so we shouldn't be force to convert them later.
- RANDOM_SEED = 42: 
when we split train_good into "fit" (used to build the model) and "val" (held back purely to pick a detection threshold), we do that split by shuffling the list of images into random order and cutting it. Without a fixed seed, "random" really means different every single time you re-run the notebook, so today's shuffle would differ from tomorrow's, and your fit/val split would silently change between runs.
That's a problem because it breaks reproducibility: if you show your baseline's score to others, then re-run the notebook tomorrow and get a slightly different score, nobody can tell if that's due to a real code change or just random luck from a different split. Setting RANDOM_SEED = 42 forces the "random" shuffle to always come out in the same order every time, same split, same results, every run, on any machine. The number 42 itself is arbitrary (it's a common reference to "The Hitchhiker's Guide to the Galaxy," where 42 is the supercomputer Deep Thought's answer to "the ultimate question of life, the universe, and everything."). So any other fixed number would work identically.
- VAL_FRACTION = 0.15:
Remember we can't pick our detection threshold using the test set, since that would mean cheating by peeking at the data we are supposed to be evaluated on. So we need a separate small slice of normal images, held out purely for choosing the threshold, that's neither used to build the model nor used for final scoring. That's what VAL_FRACTION carves out — 15% of the train_good images get set aside as this "val" set, the other 85% ("fit") is what actually builds the model.
Why 15% specifically: in our first report, we already found that each category only has roughly 200–300 normal training images total (look at Appendix E). Take screw, for example — it has around 320 training images. 15% of that is about 48 images held out for val, leaving around 270 to actually fit the model. That's enough images in val to get a stable estimate of what percentile of anomaly scores looks normal without cutting so deep into the fit set that the model has too little data left to learn from.
10% or 20% would also be defensible, but 15% is a reasonable middle ground given how small these training sets already are.

- Important: 
One trap to avoid: never pick your decision threshold using the test set.

#### Functions to load one image and one mask:

Masks are answer key for where the defect actually is in an image. Each mask is the same size as its image (256×256), but instead of colors, every pixel is either 0 (normal) or 1 (defective). We use them to check whether the model's anomaly scores line up with where the real defects are.

In [283]:
# functions to load one image and one mask (load_image, load_mask)

def load_image(path, size, category):
    mode = "L" if category in GRAYSCALE_CATEGORIES else "RGB"
    img = Image.open(path).convert(mode).resize((size, size), Image.Resampling.BILINEAR)
    return np.asarray(img, dtype=np.float32) / 255.0


def load_mask(path, size):
    img = Image.open(path).convert("L").resize((size, size), Image.Resampling.NEAREST)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return (arr > 0.5).astype(np.float32)

RGB stands for Red, Green, Blue, the three color channels a normal color photo is built from. Every pixel's color is made by mixing amounts of red, green, and blue light. So an RGB image stores 3 numbers per pixel (how much red, how much green, how much blue), while a grayscale image stores just 1 number per pixel (how bright/dark it is, no color).

#### Collecting file paths for a category:

In [284]:
# collecting file paths for a category (load_category_paths)

def load_category_paths(category):
    cat_dir = DATA_ROOT / category
    train_good = sorted((cat_dir / "train" / "good").glob("*.png"))
    test_good = sorted((cat_dir / "test" / "good").glob("*.png"))

    test_defect, test_defect_masks = [], []
    for defect_dir in sorted((cat_dir / "test").iterdir()):
        if not defect_dir.is_dir() or defect_dir.name == "good":
            continue
        for img_path in sorted(defect_dir.glob("*.png")):
            mask_path = cat_dir / "ground_truth" / defect_dir.name / f"{img_path.stem}_mask.png"
            test_defect.append(img_path)
            test_defect_masks.append(mask_path)

    return train_good, test_good, test_defect, test_defect_masks

What it does: 
for one category (it could be "screw"), collects four lists of file paths:
- train_good: every normal training image (the only thing models get to learn from).
- test_good: normal images in the test set (used to check for false alarms).
- test_defect:  every defective test image, across all defect subtypes (e.g. screw/test/scratch_head/, screw/test/thread_top/, ..., the loop walks through each defect-type subfolder).
- test_defect_masks: the matching ground-truth mask path for each defective image, built by swapping the folder for ground_truth/<defect_type>/ and appending _mask to the filename — this mirrors exactly how MVTec AD names its mask files.

### First category: screw

#### Quick check on "screw" category:

In [285]:
# Quick check: file paths for one category (screw)

train_good, test_good, test_defect, test_defect_masks = load_category_paths("screw")
print(f"train_good={len(train_good)}  test_good={len(test_good)}  test_defect={len(test_defect)}")

train_good=320  test_good=41  test_defect=119


What this check tells: it calls the function on "screw" specifically and prints how many images landed in each list. We should see numbers matching what our EDA already found for screw, around 320 training images, 41 normal test images, and 119 defective test images (per our Report 1 Appendix E in our dataset composition table). If the numbers come out different or it errors, that tells us something is off in the path-matching logic before we move any further.

#### The fit/val split function: splitting train_good into the 85% "fit" and 15% "val":

In [286]:
# the fit/val split function: splitting train_good into the 85% "fit" and 15% "val"

def split_fit_val(train_good, val_fraction=VAL_FRACTION, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    paths = list(train_good)
    rng.shuffle(paths)
    n_val = max(1, round(len(paths) * val_fraction))
    return paths[n_val:], paths[:n_val]  # fit, val

What it does: "np.random.default_rng(seed)" creates a random-number generator locked to the fixed seed (42), this is what makes the random shuffle reproducible every time it is re-run. "rng.shuffle(paths)" randomly reorders the list of image paths in place. "n_val" calculates how many images 15% represents, it rounded with max(1, ...) as a safety net so it never get zero validation images even for a tiny category. Then it slices the shuffled list into two pieces: everything after the first "n_val" images becomes "fit", the first "n_val" become "val".

#### Quick check: fit/val split:

In [287]:
# Quick check: fit/val split

fit_paths, val_paths = split_fit_val(train_good)
print(f"fit={len(fit_paths)}  val={len(val_paths)}")

fit=272  val=48


It shows fit=272 val=48 (320 total, 15% of 320 rounds to 48)

#### Batch image loading:

In [288]:
# batch image loading:

def load_stack(paths, size, category):
    return np.stack([load_image(p, size, category) for p in paths], axis=0)

Pay attention to what it does: takes a list of file paths, calls load_image function (from Step 3) on each one, and stacks all the resulting arrays together into a single NumPy array with one extra dimension on the front — so instead of 272 separate 256×256 arrays, we get one array of shape (272, 256, 256) (or (272, 256, 256, 3) for color categories). This is the format the dummy classifier and baseline will expect.

test only on val since it's small (48 images) and not all 272 fit images.

In [289]:
# test only on val 

val_images = load_stack(val_paths, SIZE, "screw")
print(val_images.shape)

(48, 256, 256)


(48, 256, 256) with no third channel dimension confirms grayscale handling is working (screw is one of your grayscale categories), and 48 matches the val count.

#### The dummy classifier (it always predicts "normal"):

In [290]:
# the dummy classifier: always predicts "normal"

def dummy_score(images):
    """Assigns every pixel an anomaly score of 0, never flags anything as anomalous."""
    if images.ndim == 4:  # color images: (N, H, W, C)
        return np.zeros(images.shape[:3], dtype=np.float32)
    return np.zeros(images.shape, dtype=np.float32)  # grayscale: (N, H, W)

What it does: no learning, no parameters, no looking at any data. Every pixel of every image gets a score of exactly 0, meaning "not anomalous." It's the simplest possible baseline, not a real candidate model.

Then use the dummy classifier on real data to see the accuracy paradox appear in our own numbers.

In [291]:
# Quick check: loading test images and masks for "screw"

test_good_images = load_stack(test_good, SIZE, "screw")
test_defect_images = load_stack(test_defect, SIZE, "screw")
test_defect_mask_arrays = np.stack([load_mask(p, SIZE) for p in test_defect_masks], axis=0)

print(f"test_good: {test_good_images.shape}  test_defect: {test_defect_images.shape}  masks: {test_defect_mask_arrays.shape}")

test_good: (41, 256, 256)  test_defect: (119, 256, 256)  masks: (119, 256, 256)


For the next stepcomputing the dummy classifier's accuracy vs its real score.

#### Dummy classifier metrics:
Dummy classifier: accuracy vs. Average Precision

Score the "always normal" classifier on real screw test data. 
Dummy classifier always returns 0.

In [292]:
dummy_good_scores = dummy_score(test_good_images)
dummy_defect_scores = dummy_score(test_defect_images)

Flatten everything into two matching 1D lists: one true label per pixel, one predicted score per pixel, in the same order.

In [293]:
y_true = np.concatenate([
    np.zeros(dummy_good_scores.size, dtype=np.float32),
    test_defect_mask_arrays.reshape(-1),
])
y_score = np.concatenate([
    dummy_good_scores.reshape(-1),
    dummy_defect_scores.reshape(-1),
])

Comparing the naive accuracy (predict normal everywhere) against the honest score, Average Precision.

In [294]:
naive_accuracy = 1.0 - y_true.mean()

y_true.mean() is the average of a bunch of 0s and 1s, which equals the fraction of pixels that are truly anomalous. Since the dummy predicts "normal" for literally every pixel, it's correct everywhere except the truly anomalous ones, so its accuracy is just "1 minus the fraction that are actually anomalous."

In [295]:
dummy_ap = average_precision_score(y_true, y_score)

This uses sklearn's built-in Average Precision function, comparing  true labels against the predicted scores. 
why this comes out so close to the anomaly rate here specifically: since every single prediction is tied at exactly 0 (no variation at all), there's no way to rank pixels as "more or less suspicious" so sklearn essentially can't distinguish anything and the score goes down to almost the base rate of positives in the data. 
That's mathematically why dummy_ap, matching almost exactly the true anomalous-pixel fraction.

In [296]:
print(f"Naive pixel accuracy (predict normal everywhere): {naive_accuracy:.4f}")
print(f"Pixel Average Precision: {dummy_ap:.4f}")

Naive pixel accuracy (predict normal everywhere): 0.9975
Pixel Average Precision: 0.0025


This is the accuracy paradox, proven in our own screw data. A classifier that does absolutely nothing, no learning, no logic and just predict normal everywhere and it looks like a near-perfect model by accuracy (99.75%!), while its actual, honest score (Average Precision) is essentially zero. It's not really 99.75% good, this accuracy just hides that fact because normal pixels so overwhelmingly dominate the count. 

That gap between 0.9975 and 0.0025 is our answer to the question, what it shows about its accuracy, it means nothing, it's an imbalance and not a sign of a working model.

remmember that our EDA found screw's anomalous-pixel ratio at ~0.34% and this run gives 0.25%, it is close but not identical, which makes sense since this includes the 41 normal test images too, which drags the overall anomaly ratio down slightly further than the defective-images-only number in our report. I is just a different denominator.

### The baseline model:

##### First baseline: per-pixel z-score vs. normal training statistics:
For each pixel location, compute the mean and standard deviation of that pixel's brightness across the normal training images ("fit" set). At test time, score a pixel by how many standard deviations away from "normal at this exact spot" it is.

- Why this fits the data: MVTec AD images are captured with a fixed camera and consistent object placement, so "what does a normal pixel at position (x, y) look like" is a learnable, meaningful question, no deep model required, and it only needs normal-only images, matching what we actually have.

- Where it's expected to fail: screw is flagged in the EDA for a train/test brightness shift, so this model may raise false alarms from lighting differences, not real defects. It also has no concept of shape or texture, only local brightness, so it will miss defects that don't change brightness much.

In [297]:
def fit_pixel_stats(fit_images):
    return {"mean": fit_images.mean(axis=0), "std": fit_images.std(axis=0)}

Quick check: computing per-pixel stats from the fit set

In [298]:
# loading the 272 fit images now (only loaded val earlier).
fit_images = load_stack(fit_paths, SIZE, "screw")

stats = fit_pixel_stats(fit_images)
print(f"fit_images: {fit_images.shape}  mean shape: {stats['mean'].shape}  std shape: {stats['std'].shape}")

fit_images: (272, 256, 256)  mean shape: (256, 256)  std shape: (256, 256)


What it does: fit_images has shape (272, 256, 256), 272 normal training images. 
mean(axis=0) averages across those 272 images, position by position, leaving a single (256, 256) array, the "average normal image." 
Same for .std(axis=0), the pixel-by-pixel standard deviation across those 272 images, showing how much natural variation exists at each spot even among normal images.

#### Scoring: how far is each pixel from "normal"?

1. Scoring function:

This is the actual baseline model: for every pixel, it asks "how many standard deviations away from the normal training average is this pixel?" Big z-score = looks unusual = likely a defect.

In [299]:
EPS = 1e-6

def pixel_zscore(images, stats):
    z = np.abs(images - stats["mean"][None, ...]) / (stats["std"][None, ...] + EPS)
    return z.astype(np.float32)

EPS = 1e-6 is a safety number added to the denominator to avoid dividing by zero. Look at the z-score formula: |pixel − mean| / (std + EPS). If some pixel has zero variation across all training images (std = 0 could happens with a perfectly uniform background), dividing by std alone would crash with a "divide by zero" error, or produce inf. Adding 1e-6 avoid this.

2. Test it on val:

This is Sanity check which is z-scores on validation images (the 48 normal images we held out earlier, theynever been touched by training or testing). 
It gives you a z-score for every pixel in every validation image, so you can see what "normal" z-scores typically look like before we use them to set a threshold.
Since val_images are all normal, their z-scores should be small.

In [300]:
val_scores = pixel_zscore(val_images, stats)
print(f"val_scores shape: {val_scores.shape}  min: {val_scores.min():.3f}  max: {val_scores.max():.3f}  mean: {val_scores.mean():.3f}")

val_scores shape: (48, 256, 256)  min: 0.000  max: 12.333  mean: 0.753


Mean around 0.75 makes sense (most pixels close to their normal training average), and the high max (12.3) is normal too, a few pixels in the image barely change at all across the training images (like flat background), so their normal "spread" (std) is almost zero. Any tiny noise on those pixels gets divided by that tiny number and blows up into a big z-score, even though nothing is actually wrong. That's exactly why we don't just look at raw pixels and guess a cutoff by eye, we calculate the threshold statistically, from this validation distribution.

3. Threshold function from validation scores only:

In [301]:
def choose_threshold(val_scores, percentile=99.0):
    return float(np.percentile(val_scores.reshape(-1), percentile))

- This defines the threshold rule: 

take all the z-scores from the normal validation images, and pick the value below which 99% of them fall. Anything above that, in the real test set, we'll call "anomalous." This is the "never touch the test set to pick a threshold" rule, the threshold comes entirely from val_scores, which are normal-only and never scored against test labels.

- Why the 99th percentile for the threshold:

The threshold percentile directly controls the trade-off between false alarms and missed
defects. Setting it at the 99th percentile of normal validation scores means roughly 1% of
truly normal pixels are expected to be flagged as false positives.

We chose 99th percentile because in a real production setting, flagging a normal part as
defective (a false alarm) has a direct cost, wasted inspection time, unnecessary rework, or
discarding good parts. Missing an actual defect (a false negative) also has a cost, but we
treat false alarms as the higher priority to minimize for a first baseline, since it's easier to
justify a stricter model to a production line than one that get false alarm too often.

- This percentile is not a fixed, "correct" choice , it should ultimately be set based on the
actual acceptable false-positive rate for the inspection line in question, which is a business
decision as much as a modeling one. We can revisit and tune it once that information is
available.

4. Pick the threshold:

In [302]:
threshold = choose_threshold(val_scores, percentile=99.0)
print(f"Threshold (99th percentile of normal val scores): {threshold:.4f}")

Threshold (99th percentile of normal val scores): 3.0930


We now have everything needed to actually test the baseline on the real test set. First cell, score the test images with the same z-score function:

5. Score the real test images with the baseline:

In [303]:
baseline_good_scores = pixel_zscore(test_good_images, stats)
baseline_defect_scores = pixel_zscore(test_defect_images, stats)

This applies our trained "normal" statistics to the actual test images (both test_good_images and test_defect_images), same function, same stats, just on data the model has never seen before.

Building the score array to compare against the same y_true from the dummy classifier step:

In [304]:
baseline_y_score = np.concatenate([
    baseline_good_scores.reshape(-1),
    baseline_defect_scores.reshape(-1),
])

We reuse the same y_true from before (it hasn't changed: same images, same masks). This just lines up the baseline's scores in the same order as those labels, so we can compare them fairly.

the final piece that actually computes the baseline's Average Precision, and accuracy.

this actually scores the baseline:

In [305]:
baseline_ap = average_precision_score(y_true, baseline_y_score)
baseline_pred = (baseline_y_score > threshold).astype(np.float32)
baseline_accuracy = accuracy_score(y_true, baseline_pred)

print(f"Baseline Pixel Average Precision: {baseline_ap:.4f}")
print(f"Baseline Pixel accuracy at threshold: {baseline_accuracy:.4f}")

Baseline Pixel Average Precision: 0.0126
Baseline Pixel accuracy at threshold: 0.9822


Two numbers, two roles:

- Average Precision is the main score, the one to report. Directly comparable to the dummy classifier's 0.0025.
- Accuracy at threshold uses the 3.0930 cutoff from validation (never from test) — this shows how the model performs in practice, without peeking at test labels to pick the cutoff.

These numbers tell a good story:

Average Precision went from 0.0025 (dummy) to 0.0126. That's roughly 5x better than random guessing at this task — small in absolute terms (pixel-level anomaly detection is genuinely hard), but real signal, not noise.

Now the interesting part: accuracy actually dropped, from the dummy's 0.9975 down to 0.9822. This is your accuracy paradox finding from Report 1, showing up again here in the modeling phase. The dummy classifier "wins" on accuracy only because it never flags anything — pure artifact of imbalance. Your real baseline is doing actual work (flagging some anomalous pixels), which costs it a bit of raw accuracy while massively improving actual detection ability (AP). This is a strong, concrete point for your report: it shows why accuracy is a bad metric here, not just in th

## Summary: dummy classifier vs. first baseline

The dummy classifier ("always normal") reaches 99.75% pixel accuracy but an Average Precision
of only 0.0025 — this is the accuracy paradox: high accuracy with essentially zero real
detection ability, caused by severe class imbalance (most pixels are normal).

The per-pixel z-score baseline reaches a lower accuracy (98.22%) but a much higher Average
Precision (0.0126, ~5x the dummy). Accuracy alone would suggest the baseline is "worse" than the dummy classifier, this is exactly why we use Average Precision as our primary metric and treat accuracy and

The threshold (3.093) was chosen from the held-out validation split (normal images only,
99th percentile of their z-scores) and never touched the test set, satisfying the
never-threshold-on-test rule.

### Second category: bottle

collects the file paths for the bottle category:

In [306]:
bottle_train_good, bottle_test_good, bottle_test_defect, bottle_test_defect_masks = load_category_paths("bottle")
print(f"train_good={len(bottle_train_good)}  test_good={len(bottle_test_good)}  test_defect={len(bottle_test_defect)}")

train_good=209  test_good=20  test_defect=63


here test set is 83 (20 normal + 63 anomaly)

split off a validation set:

In [307]:
bottle_fit_paths, bottle_val_paths = split_fit_val(bottle_train_good)
print(f"fit={len(bottle_fit_paths)}  val={len(bottle_val_paths)}")

fit=178  val=31


loading the actual images (fit + validation sets):

In [308]:
bottle_fit_images = load_stack(bottle_fit_paths, SIZE, "bottle")
bottle_val_images = load_stack(bottle_val_paths, SIZE, "bottle")
print(f"fit_images: {bottle_fit_images.shape}  val_images: {bottle_val_images.shape}")

fit_images: (178, 256, 256, 3)  val_images: (31, 256, 256, 3)


RGB stands for Red, Green, Blue, the three color channels a normal color photo is built from. Every pixel's color is made by mixing amounts of red, green, and blue light. So an RGB image stores 3 numbers per pixel (how much red, how much green, how much blue), while a grayscale image stores just 1 number per pixel (how bright/dark it is, no color). That's the "3" in the shape (31, 256, 256, 3), meaning 3 color values per pixel. So (31, 256, 256, 3) means 31 images, each 256×256 pixels, and each pixel holds 3 values instead of 1. 
Since bottle isn't in our grayscale category list, load_image kept all 3 color channels.

In [309]:
bottle_test_good_images = load_stack(bottle_test_good, SIZE, "bottle")
bottle_test_defect_images = load_stack(bottle_test_defect, SIZE, "bottle")
bottle_test_defect_mask_arrays = np.stack([load_mask(p, SIZE) for p in bottle_test_defect_masks], axis=0)
print(f"test_good: {bottle_test_good_images.shape}  test_defect: {bottle_test_defect_images.shape}  masks: {bottle_test_defect_mask_arrays.shape}")

test_good: (20, 256, 256, 3)  test_defect: (63, 256, 256, 3)  masks: (63, 256, 256)


Masks stay single-channel (63, 256, 256) even for RGB images.
ground-truth masks are always just "defect or not," so no color needed there.

fit the "normal" statistics (mean/std per pixel) on the bottle fit set:

In [310]:
bottle_stats = fit_pixel_stats(bottle_fit_images)
print(f"mean shape: {bottle_stats['mean'].shape}  std shape: {bottle_stats['std'].shape}")

mean shape: (256, 256, 3)  std shape: (256, 256, 3)


since bottle images are RGB, the mean and std are now stored per pixel per color channel (256×256×3 instead of 256×256). Same idea as before, just one extra dimension for color.

test the z-score function on bottle_val_images:

In [311]:
bottle_val_scores = pixel_zscore(bottle_val_images, bottle_stats)
print(f"val_scores shape: {bottle_val_scores.shape}  min: {bottle_val_scores.min():.3f}  max: {bottle_val_scores.max():.3f}  mean: {bottle_val_scores.mean():.3f}")

val_scores shape: (31, 256, 256, 3)  min: 0.000  max: 82352.938  mean: 1.456


- Red Flag: This max value (82,352) is much more compare to screw, where the max was only 12.3. Something different is happening with bottle.

- EPS = 1e-6 being this tiny is also why we see this huge max z-score (82,352) on bottle (the z-score formula: |pixel − mean| / (std + EPS)).  EPS stops the code from crashing, but if std is very close to zero (not exactly zero), you still get a huge, unstable z-score. EPS prevents an error, it doesn't prevent an extreme value.

threshold for bottle:

- Threshold is computed on the channel-averaged z-scores to match how test scores are combined, soomputing it on raw per-channel values would miscalibrate the false-positive rate. 
- The fix is combining the 3 color channels into one score first, exactly the same way you already do for the test scores, before picking the threshold

In [312]:
bottle_val_scores_combined = bottle_val_scores.mean(axis=-1)
bottle_threshold = choose_threshold(bottle_val_scores_combined, percentile=99.0)
print(f"Threshold (99th percentile of normal val scores): {bottle_threshold:.4f}")

Threshold (99th percentile of normal val scores): 2.7997


the threshold itself (2.88) is totally reasonable, close to screw's (3.09), even though the max was over 82,000. That confirms the 99th-percentile approach is robust to those few extreme background pixels. this is exactly why we use percentile-based thresholds instead of raw min/max.

score the actual test images for bottle:

In [313]:
bottle_baseline_good_scores = pixel_zscore(bottle_test_good_images, bottle_stats)
bottle_baseline_defect_scores = pixel_zscore(bottle_test_defect_images, bottle_stats)

Before we can compare these scores to the masks, there's one more step needed for bottle that didn't come up with screw because screw was grayscale.

Our masks only say "defect or not" per pixel, it is one value each for shape (256, 256). But bottle scores have 3 numbers per pixel (one z-score per color channel) for shape (256, 256, 3). We need to combine those 3 into a single "how unusual is this pixel" number before we can compare to the mask here:

In [314]:
bottle_baseline_good_combined = bottle_baseline_good_scores.mean(axis=-1)
bottle_baseline_defect_combined = bottle_baseline_defect_scores.mean(axis=-1)
print(f"good: {bottle_baseline_good_combined.shape}  defect: {bottle_baseline_defect_combined.shape}")

good: (20, 256, 256)  defect: (63, 256, 256)


This averages the red, green, and blue z-scores at each pixel into one number — the pixel's overall "distance from normal color." Simple and explainable, same way as everything else was upto now. 

taking the max instead would be more sensitive but also more exposed to that background-noise blow-up we just saw. Here, mean value is the safer choice.

mask shape:

In [315]:
bottle_y_true = np.concatenate([
    np.zeros(bottle_baseline_good_combined.size, dtype=np.float32),
    bottle_test_defect_mask_arrays.reshape(-1),
])
bottle_y_score = np.concatenate([
    bottle_baseline_good_combined.reshape(-1),
    bottle_baseline_defect_combined.reshape(-1),
])

This has ame idea as before. It lines up every pixel's true label (0 = normal, 1 = defect, from the masks) with its predicted anomaly score, in matching order.

#### Bottle baseline results (pixel-level Average Precision and accuracy)

In [316]:
bottle_ap = average_precision_score(bottle_y_true, bottle_y_score)
bottle_pred = (bottle_y_score > bottle_threshold).astype(np.float32)
bottle_accuracy = accuracy_score(bottle_y_true, bottle_pred)

print(f"Baseline Pixel Average Precision: {bottle_ap:.4f}")
print(f"Baseline Pixel accuracy at threshold: {bottle_accuracy:.4f}")

Baseline Pixel Average Precision: 0.5031
Baseline Pixel accuracy at threshold: 0.9496


Big jump compared to screw: AP=Average Precision 0.50 vs. 0.0126 On bottle.

Remember it's solving the harder task (exact pixel location, not just yes/no per image) with by far the simplest method, no deep learning, no pretrained features, just raw pixel statistics, resulted AP=0.50, which is good.

Why the huge gap between screw and bottle for the same method? Maybe because bottle defects (cracks, contamination, liquid-level issues) tend to be large, visually obvious color/shape changes against a fairly uniform bottle+background. This is exactly what raw pixel color deviation can catch. Screw defects are probably subtler geometric changes (scratch, thread damage) sitting inside naturally noisy metal texture, so the "normal" variance is already high and swallows the signal. This baseline's performance is highly category-dependent, which matters for the question "how good is the simple baseline", and it's not one number, it varies a lot by defect type.

#### Summary: baseline results across two categories : Screw and Bottle

The per-pixel z-score baseline was tested on two categories with very different results:

| Category | Dummy AP | Baseline AP | Baseline accuracy |
|----------|----------|--------------|------------------|
| screw    | 0.0025   | 0.0126       | 0.9822           |
| bottle   | (not run)| 0.5031       | 0.9496           |

The baseline improves massively over the dummy classifier in both cases, but the size of that
improvement is very category-dependent. On bottle, the simple z-score baseline reaches a
genuinely strong Average Precision (0.50), likely because bottle defects tend to be large,
visually obvious color/shape changes against a fairly uniform background. On screw, the
improvement is real but much smaller (AP 0.0126), likely because screw defects are subtler
geometric changes sitting inside naturally noisy metal texture, which raises the "normal"
variance and makes small deviations harder to detect with raw pixel statistics.

This shows that even a very simple baseline is not equally good or bad everywhere, its
usefulness depends heavily on the category's defect type and background uniformity, which is
relevant when comparing it against the deep-feature-based models: whole-image
ResNet-18 + Nearest Neighbors or PatchCore. Since those may show a different pattern
of category-dependence.

As with screw, both thresholds (2.7997 for bottle, 3.0930 for screw) were chosen only from
held-out normal validation data, never from the test set.